# Eignungsprüfung: Eurovision Song Contest (ESC) Datensatz

Dieses Notebook prüft, ob der ESC-Datensatz für die Aufgabe *Data Exploration* tauglich ist.
Es folgt der Struktur von `Template_DataExploration.ipynb` (Fragen 1–4, 8, 9) und endet mit einem Fazit.
Die eigentliche Abgabe (Fragen 5–7, 10, 11) wird im Template ausgearbeitet.

## 1) Datensatz laden

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

contestants = pd.read_csv("data/contestants.csv")      # ein Beitrag (Land x Jahr)
votes       = pd.read_csv("data/votes.csv")            # Punkte von Land A an Land B
betting     = pd.read_csv("data/betting_offices.csv")  # Buchmacher-Quoten (optional)

contestants.head(3)

,year,to_country_id,to_country,performer,song,place_contest,sf_num,running_final,running_sf,place_final,points_final,place_sf,points_sf,points_tele_final,points_jury_final,points_tele_sf,points_jury_sf,composers,lyricists,lyrics,youtube_url
0,1956,ch,Switzerland,Lys Assia,Refrain,2.0,NaN,2.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Georg Benz Stahl,NaN,"(Refrain d'amour...)\n\nRefrain, couleur du ci...",https://youtube.com/watch?v=IyqIPvOkiRk
1,1956,nl,Netherlands,Jetty Paerl,De Vogels Van Holland,2.0,NaN,1.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cor Lemaire,Annie M. G. Schmidt,De vogels van Holland zijn zo muzikaal\nZe ler...,https://youtube.com/watch?v=u45UQVGRVPA
2,1956,be,Belgium,Fud Leclerc,Messieurs Les Noyés De La Seine,2.0,NaN,3.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Jacques Say;Jean Miret,Robert Montal,Messieurs les noyés de la Seine\nOuvrez-moi le...,https://youtube.com/watch?v=U9O3sqlyra0


## 2) Kurzbeschreibung & Quelle

**Inhalt:** Alle Beiträge des Eurovision Song Contest von 1956 bis 2023 (Land, Interpret:in, Titel,
Startnummer, Platzierung und Punkte in Semifinale/Finale, seit 2016 getrennt nach Jury und Televoting,
Komponist:innen, Textdichter:innen, Songtext, YouTube-Link) sowie die **vollständige Punktematrix**
(welches Land hat welchem Land wie viele Punkte gegeben) und Buchmacher-Quoten ab 2015.

**Quelle:** <https://github.com/Spijkervet/eurovision-dataset> (Release `2023`,
Dateien `contestants.csv`, `votes.csv`, `betting_offices.csv`), gescraped von eurovision.tv / Wikipedia.
Lizenz: MIT.

## 3) Größe des Datensatzes (Zeilen / Spalten)

In [2]:
for name, df in [("contestants", contestants), ("votes", votes), ("betting", betting)]:
    print(f"{name:12s} {df.shape[0]:6d} Zeilen x {df.shape[1]:2d} Spalten")

print("\nJahre  :", contestants.year.min(), "-", contestants.year.max(),
      f"({contestants.year.nunique()} Ausgaben, 2020 abgesagt)")
print("Laender:", contestants.to_country.nunique(), "(inkl. historischer Staaten)")
print("\nSpalten contestants:\n", list(contestants.columns))
print("\nSpalten votes:\n", list(votes.columns))

contestants    1734 Zeilen x 21 Spalten
votes         51354 Zeilen x  9 Spalten
betting        9453 Zeilen x 11 Spalten

Jahre  : 1956 - 2023 (68 Ausgaben, 2020 abgesagt)
Laender: 54 (inkl. historischer Staaten)

Spalten contestants:
 ['year', 'to_country_id', 'to_country', 'performer', 'song', 'place_contest', 'sf_num', 'running_final', 'running_sf', 'place_final', 'points_final', 'place_sf', 'points_sf', 'points_tele_final', 'points_jury_final', 'points_tele_sf', 'points_jury_sf', 'composers', 'lyricists', 'lyrics', 'youtube_url']

Spalten votes:
 ['year', 'round', 'from_country_id', 'to_country_id', 'from_country', 'to_country', 'total_points', 'tele_points', 'jury_points']


In [3]:
# Teilnehmerfeld pro Jahr - zeigt das Wachstum des Contests
size_per_year = contestants.groupby("year").size()
print(size_per_year.head(3).to_string(), "\n...\n", size_per_year.tail(5).to_string())

year
1956    12
1957    10
1958    10 
...
 year
2019    41
2020    41
2021    39
2022    40
2023    52


**Relevante Spalten** (multivariat, ca. 10 auswertbare Attribute):
`year`, `to_country` / `to_country_id`, `performer`, `song`, `running_final`, `place_contest`,
`place_final`, `points_final`, `points_tele_final`, `points_jury_final`, `sf_num`, `place_sf`, `points_sf`.
`lyrics`, `youtube_url`, `composers`, `lyricists` sind für die Visualisierungen nicht nötig und können
mit `drop` entfernt werden. Aus `votes.csv` sind alle 9 Spalten relevant (Matrix-/Netzwerksicht).

## 4) Fehlende Werte & Datenqualität

In [4]:
miss = (contestants.isna().mean()*100).round(1).sort_values(ascending=False)
print("Fehlende Werte contestants (%):\n", miss.to_string())
print("\nFehlende Werte votes (%):\n",
      (votes.isna().mean()*100).round(1).sort_values(ascending=False).to_string())

Fehlende Werte contestants (%):
 points_jury_final    89.6
points_tele_final    89.6
points_jury_sf       87.8
points_tele_sf       87.8
running_sf           65.1
place_sf             65.1
points_sf            65.1
sf_num               63.1
lyricists            43.2
points_final         20.1
running_final        19.4
place_final          19.4
place_contest         3.2
composers             3.1
lyrics                0.7
youtube_url           0.7
song                  0.2
performer             0.0
to_country            0.0
to_country_id         0.0
year                  0.0

Fehlende Werte votes (%):
 jury_points        77.2
tele_points        77.1
year                0.0
round               0.0
from_country_id     0.0
from_country        0.0
to_country_id       0.0
total_points        0.0
to_country          0.0


In [5]:
# Warum fehlen Werte? -> systematisch, nicht zufaellig (Regelaenderungen des Contests)
print("Anteil vorhandener Werte je Jahrzehnt:")
tmp = contestants.assign(dekade=(contestants.year//10)*10)
print(tmp.groupby("dekade")[["points_final","place_final","running_final",
                             "sf_num","points_tele_final","points_jury_final"]]
        .apply(lambda d: d.notna().mean().round(2)).to_string())
print("\n2020: Contest abgesagt ->", (contestants.year==2020).sum(),
      "Beitraege, davon mit Punkten:", contestants[contestants.year==2020].points_final.notna().sum())

Anteil vorhandener Werte je Jahrzehnt:
        points_final  place_final  running_final  sf_num  points_tele_final  points_jury_final
dekade                                                                                        
1950            0.72         1.00           1.00    0.00               0.00               0.00
1960            1.00         1.00           1.00    0.00               0.00               0.00
1970            1.00         1.00           1.00    0.00               0.00               0.00
1980            1.00         1.00           1.00    0.00               0.00               0.00
1990            1.00         1.00           1.00    0.00               0.00               0.00
2000            0.72         0.72           0.72    0.51               0.00               0.00
2010            0.63         0.63           0.63    0.86               0.25               0.25
2020            0.45         0.45           0.45    0.69               0.45               0.45

2020: Cont

### Problem 1: doppelte Zeilen

In [6]:
dups = contestants[contestants.duplicated(["year","to_country_id"], keep=False)]
print("Doppelte (Jahr, Land)-Zeilen:", len(dups), "in den Jahren", sorted(dups.year.unique()))

print("\n1956: ECHTE Doppelbeitraege (jedes Land sang zwei Lieder) - korrekt, nicht loeschen:")
print(dups[dups.year==1956][["year","to_country","performer","song"]].head(4).to_string(index=False))

print("\n2023: SCRAPING-ARTEFAKT - jede Zeile existiert zweimal, die zweite Kopie ist leer:")
print(dups[(dups.year==2023) & (dups.to_country=="Sweden")]
      [["year","to_country","performer","song","place_contest","points_final","running_final"]]
      .to_string(index=False))
print("\n2023 -> Zeilen:", (contestants.year==2023).sum(), "statt der tatsaechlichen 37 Beitraege")

Doppelte (Jahr, Land)-Zeilen: 40 in den Jahren [np.int64(1956), np.int64(2023)]

1956: ECHTE Doppelbeitraege (jedes Land sang zwei Lieder) - korrekt, nicht loeschen:
 year  to_country              performer                            song
 1956 Netherlands            Jetty Paerl           De Vogels Van Holland
 1956     Belgium            Fud Leclerc Messieurs Les Noyés De La Seine
 1956     Germany Walter Andreas Schwarz   Im Wartesaal Zum Großen Glück
 1956      France           Mathé Altéry                  Le Temps Perdu

2023: SCRAPING-ARTEFAKT - jede Zeile existiert zweimal, die zweite Kopie ist leer:
 year to_country performer   song  place_contest  points_final  running_final
 2023     Sweden    Loreen Tattoo            1.0         583.0            9.0
 2023     Sweden    Loreen Tattoo            NaN           NaN            NaN

2023 -> Zeilen: 52 statt der tatsaechlichen 37 Beitraege


### Problem 2: uneinheitliche Landesbezeichnungen

In [7]:
conflicts = {k: list(x) for k, x in contestants.groupby("to_country_id").to_country.unique().items()
             if len(x) > 1}
print("Ein Code, mehrere Namen:", conflicts)
print("\nHistorische Codes in votes:", [c for c in ["yu","cs","wld"] if c in set(votes.from_country_id)],
      "(yu=Jugoslawien, cs=Serbien&Montenegro, wld='Rest of the World'-Televote ab 2023)")

Ein Code, mehrere Namen: {'cz': ['Czech Republic', 'Czechia'], 'mk': ['North Macedonia', 'North MacedoniaN.Macedonia']}

Historische Codes in votes: ['yu', 'cs', 'wld'] (yu=Jugoslawien, cs=Serbien&Montenegro, wld='Rest of the World'-Televote ab 2023)


### Problem 3: `to_country_id` enthält teilweise Ländernamen statt Ländercodes

In [8]:
isname = contestants.to_country_id.str.len() > 2
print("Zeilen mit Namen statt Code:", isname.sum(), "von", len(contestants))
print("betroffene Jahre:", contestants[isname].year.min(), "-", contestants[isname].year.max())
print("davon mit points_final:", contestants[isname].points_final.notna().sum(),
      "-> betrifft ausschliesslich im Semifinale ausgeschiedene Beitraege\n")
print(contestants[contestants.year==2004]
      [["to_country_id","to_country","place_contest","place_final","points_final"]]
      .tail(6).to_string(index=False))
print("\n=> Ein naiver Join contestants<->votes ueber to_country_id verliert diese 259 Zeilen still!")

Zeilen mit Namen statt Code: 259 von 1734
betroffene Jahre: 2004 - 2020
davon mit points_final: 0 -> betrifft ausschliesslich im Semifinale ausgeschiedene Beitraege

to_country_id  to_country  place_contest  place_final  points_final
       Latvia      Latvia           31.0          NaN           NaN
      Andorra     Andorra           32.0          NaN           NaN
      Belarus     Belarus           33.0          NaN           NaN
       Monaco      Monaco           33.0          NaN           NaN
     Slovenia    Slovenia           35.0          NaN           NaN
  Switzerland Switzerland           36.0          NaN           NaN

=> Ein naiver Join contestants<->votes ueber to_country_id verliert diese 259 Zeilen still!


### Bereinigung (Vorschlag)

In [9]:
def bereinigen(df):
    df = df.copy()
    # 1) einheitlicher Laendercode
    isname = df.to_country_id.str.len() > 2
    lookup = (df[~isname].drop_duplicates("to_country")
                .set_index("to_country").to_country_id.to_dict())
    lookup.update({"Andorra": "ad"})           # war nie im Finale -> kein Code vorhanden
    df["country_id"] = df.to_country_id.where(~isname, df.to_country.map(lookup))
    # 2) einheitlicher Laendername
    df["country"] = df.to_country.replace({"Czechia": "Czech Republic",
                                           "North MacedoniaN.Macedonia": "North Macedonia"})
    # 3) Dubletten 2023 entfernen (Zeile mit Ergebniswerten behalten), 1956 bewusst ausnehmen
    art = df[df.year == 2023].sort_values("place_contest").drop_duplicates(["year","country_id"])
    df = pd.concat([df[df.year != 2023], art]).sort_values(["year","place_contest"])
    return df

esc = bereinigen(contestants)
print("Zeilen nach Bereinigung:", len(esc), "| 2023:", (esc.year==2023).sum(),
      "| ungemappte Codes:", esc.country_id.isna().sum(),
      "| eindeutige Codes:", esc.country_id.nunique())

Zeilen nach Bereinigung: 1719 | 2023: 37 | ungemappte Codes: 0 | eindeutige Codes: 52


## Konsistenzprüfung: passen `votes` und `contestants` zusammen?

In [10]:
vsum = (votes[votes["round"]=="final"]
         .groupby(["year","to_country_id"]).total_points.sum().rename("summe_aus_votes"))
check = esc.merge(vsum, left_on=["year","country_id"], right_index=True, how="left")
check = check[check.points_final.notna()]
print("Finalergebnisse gesamt:", len(check),
      "| in votes wiedergefunden:", check.summe_aus_votes.notna().sum())
print("Uebereinstimmung points_final == Summe der Einzelvoten:",
      f"{(check.points_final == check.summe_aus_votes).mean():.1%}")
print("Duplikate im Votes-Schluessel (year, round, from, to):",
      votes.duplicated(["year","round","from_country_id","to_country_id"]).sum())

Finalergebnisse gesamt: 1385 | in votes wiedergefunden: 1385
Uebereinstimmung points_final == Summe der Einzelvoten: 100.0%
Duplikate im Votes-Schluessel (year, round, from, to): 0


Nach der Bereinigung sind beide Tabellen **vollständig und konsistent** verknüpfbar:
die Summe der Einzelvoten ergibt in 100 % der 1.385 Finalergebnisse exakt die Gesamtpunktzahl,
und die Votes-Tabelle enthält keine Duplikate.

**Zusammenfassung Datenqualität:**

| Problem | Ursache | Umgang |
|---|---|---|
| `points_tele_*`, `points_jury_*` zu ~90 % leer | Jury-/Televoting getrennt erst ab 2016 veröffentlicht | für Jury-vs-Publikum-Fragen auf 2016–2023 filtern |
| `sf_num`, `place_sf`, `points_sf` zu ~65 % leer | Semifinale gibt es erst ab 2004 | Semifinal-Analysen auf ≥ 2004 filtern |
| `points_final`, `place_final` zu ~20 % leer | im Semifinale ausgeschieden (+ 2020 abgesagt) | getrennt betrachten: „im Finale“ vs. „ausgeschieden“; für Gesamtplatzierung `place_contest` nutzen |
| 15 Dubletten in 2023 | Scraping-Artefakt | `drop_duplicates` (siehe `bereinigen()`) |
| 259 Zeilen mit Namen statt Code in `to_country_id` | Scraping-Artefakt (ausgeschiedene Semifinalisten) | Name → Code mappen, sonst stiller Datenverlust beim Join |
| `Czechia`/`Czech Republic`, `North MacedoniaN.Macedonia` | uneinheitliche Schreibweise | Namen vereinheitlichen |
| Punkteskala nicht vergleichbar (max. 12 bis 2015, 24 ab 2016; Feldgröße 12 → 43) | Regeländerungen | für Zeitvergleiche normalisieren (erhaltene / maximal mögliche Punkte) |

Die Lücken sind **systematisch und erklärbar** (Regeländerungen des Contests), die drei
Scraping-Artefakte sind mit wenigen Zeilen Code reparierbar.

## 8) Attributtypen

In [11]:
quant = ["year","running_final","place_contest","place_final","points_final",
         "points_tele_final","points_jury_final","place_sf","points_sf"]
print("QUANTITATIV / ORDINAL (Wertebereich):")
for c in quant:
    print(f"  {c:20s} min={esc[c].min():8.0f}  max={esc[c].max():8.0f}")

print("\nNOMINAL (Anzahl eindeutiger Klassen):")
for c in ["country","country_id","performer","song"]:
    print(f"  {c:20s} {esc[c].nunique():5d} Klassen")
print(f"  votes.round          {votes['round'].nunique():5d} Klassen -> {sorted(votes['round'].unique())}")
print(f"  votes.from_country_id{votes.from_country_id.nunique():5d} Klassen")
print(f"\n  sf_num (ordinal)     {sorted(esc.sf_num.dropna().unique())}")
print(f"  votes.total_points   {votes.total_points.min()} - {votes.total_points.max()}"
      f"  (bis 2015 max. 12, ab 2016 max. 24 = Jury + Televoting)")

QUANTITATIV / ORDINAL (Wertebereich):
  year                 min=    1956  max=    2023
  running_final        min=       1  max=      27
  place_contest        min=       1  max=      43
  place_final          min=       1  max=      27
  points_final         min=       0  max=     758
  points_tele_final    min=       0  max=     439
  points_jury_final    min=       0  max=     382
  place_sf             min=       1  max=      28
  points_sf            min=       0  max=     403

NOMINAL (Anzahl eindeutiger Klassen):
  country                 52 Klassen
  country_id              52 Klassen
  performer             1591 Klassen
  song                  1677 Klassen
  votes.round              4 Klassen -> ['final', 'semi-final', 'semi-final-1', 'semi-final-2']
  votes.from_country_id   53 Klassen

  sf_num (ordinal)     [np.float64(0.0), np.float64(1.0), np.float64(2.0)]
  votes.total_points   0 - 24  (bis 2015 max. 12, ab 2016 max. 24 = Jury + Televoting)


| Spalte | Typ | Wertebereich / Klassen |
|---|---|---|
| `year` | quantitativ (diskret), **temporal** | 1956–2023, 68 Ausgaben |
| `country` / `country_id` | nominal, **geografisch** | 52 Länder = 52 ISO-2-Codes (nach Bereinigung; inkl. Jugoslawien, Serbien & Montenegro) |
| `performer`, `song` | nominal (Identifikator) | ~1.590 / ~1.680 Klassen |
| `place_contest` | **ordinal** | 1–43 (Gesamtplatzierung inkl. Semifinale) |
| `place_final`, `place_sf` | **ordinal** | 1–27 bzw. 1–28 |
| `running_final`, `running_sf` | ordinal (Startnummer) | 1–27 |
| `sf_num` | nominal/ordinal | 0, 1, 2 (Semifinale) |
| `points_final`, `points_sf` | quantitativ (ratio) | 0–758 bzw. 0–403 |
| `points_tele_final`, `points_jury_final` | quantitativ (ratio) | 0–439 bzw. 0–382 |
| `votes.total_points` | quantitativ (ratio) | 0–24 |
| `votes.round` | nominal | 4 Klassen (final, semi-final, semi-final-1/-2) |

## 9) Weitere semantische Strukturen

In [12]:
print("TEMPORAL    : 68 Contest-Jahre, durchgehende Zeitreihe je Land")
print("GEOGRAFISCH :", esc.country_id.nunique(),
      "ISO-2-Laendercodes -> direkt fuer Karten/Choroplethen nutzbar")
print("NETZWERK    :", votes.groupby(["from_country_id","to_country_id"]).ngroups,
      "gerichtete Laenderpaare -> Matrix / Chord-Diagramm moeglich")
print("HIERARCHISCH: Contest > Runde (Semifinale 1/2, Finale) > Beitrag; Land > Jahr > Beitrag")

# Beleg, dass die Netzwerkstruktur echte Muster enthaelt: 'Nachbarschaftsvoting'
fin = votes[(votes["round"]=="final") & (votes.year>=1975)]
pair = fin.groupby(["from_country","to_country"]).total_points.agg(["mean","size"])
print("\nHoechste durchschnittliche Punktevergabe (>=10 gemeinsame Jahre):")
print(pair[pair["size"]>=10].sort_values("mean", ascending=False).head(6).round(2).to_string())

TEMPORAL    : 68 Contest-Jahre, durchgehende Zeitreihe je Land
GEOGRAFISCH : 52 ISO-2-Laendercodes -> direkt fuer Karten/Choroplethen nutzbar
NETZWERK    : 2509 gerichtete Laenderpaare -> Matrix / Chord-Diagramm moeglich
HIERARCHISCH: Contest > Runde (Semifinale 1/2, Finale) > Beitrag; Land > Jahr > Beitrag

Hoechste durchschnittliche Punktevergabe (>=10 gemeinsame Jahre):
                          mean  size
from_country to_country             
al           it          15.00    12
ro           md          14.00    13
mk           rs          13.64    11
hr           rs          12.91    11
gr           cy          12.85    27
cy           gr          12.71    31


Es stecken **vier** nutzbare Strukturen im Datensatz: *temporal* (68 Jahre),
*geografisch* (ISO-Ländercodes → Karten), *relational/Netzwerk* (≈ 2.500 gerichtete Länderpaare →
Matrix, Chord-Diagramm) und *hierarchisch* (Contest → Runde → Beitrag). Der Ausschnitt oben zeigt
bereits das bekannte „Nachbarschafts-/Blockvoting“ (z. B. Griechenland ↔ Zypern ≈ 12,8 von 12 möglichen
Punkten pro Jahr, ab 2016 von 24) – ein Muster, das sich sinnvoll **nur visuell** analysieren lässt.

*Hinweis:* In `votes.csv` enthalten die Spalten `from_country`/`to_country` ebenfalls die Ländercodes
(nicht die Namen) – für Beschriftungen also die Namen aus `contestants` dazu joinen.

---
## Fazit: Ist der Datensatz tauglich?

**Ja – der Datensatz erfüllt alle Anforderungen der Aufgabe.**

| Kriterium | Bewertung |
|---|---|
| Größe | 1.719 Beiträge + 51.354 Einzelvoten – groß genug, aber überschaubar |
| Multivariat | 21 bzw. 9 Spalten, ca. 12 inhaltlich auswertbare Attribute |
| Attributtypen | nominal, ordinal **und** quantitativ vorhanden (Voraussetzung für Frage 8) |
| Semantische Strukturen | temporal + geografisch + Netzwerk + hierarchisch (Frage 9) |
| Datentransformation nötig | ja, sinnvoll: Bereinigung, Join `contestants` ↔ `votes`, Normalisierung (Frage 7) |
| Fragen mit visueller Analyse | ja: Blockvoting, Jury vs. Publikum, Startnummer-Effekt, Entwicklung über Zeit (Frage 5/10) |
| Datenqualität | nach Bereinigung konsistent (100 % Übereinstimmung Punkte ↔ Einzelvoten) |
| Zugang | frei, MIT-Lizenz, CSV, kein Login, keine personenbezogenen Daten |

**Vor der Analyse zu erledigen:**
1. `bereinigen()` anwenden: Ländercodes vereinheitlichen, 15 Dubletten aus 2023 entfernen, Namen mappen.
2. Regeländerungen beachten (Punkteskala ab 2016, Semifinale ab 2004, Feldgröße 12 → 43) →
   für Zeitvergleiche normalisieren oder Zeitraum einschränken.
3. Jury-/Televoting-Fragen nur für 2016–2023 beantwortbar (7 Ausgaben, ca. 280 Beiträge – ausreichend).
4. 2020 (abgesagter Contest, 41 Beiträge ohne Ergebnis) in Ergebnisanalysen ausschließen.

**Mögliche Fragestellungen für Frage 5** (alle brauchen visuelle Analyse, nicht nur min/max):
- Gibt es Blockbildung/Nachbarschaftsvoting? (Matrix, Chord-Diagramm, Karte)
- Wie weit gehen Jury und Publikum auseinander, und bei welchen Beiträgen am stärksten? (Scatter, Steigungsdiagramm)
- Beeinflusst die Startnummer das Ergebnis? (Verteilung Startnummer × Platzierung über die Jahre)
- Wie hat sich die Wettbewerbsdichte über 68 Jahre entwickelt? (normalisierte Punkte, Zeitreihen)
- Sagen Buchmacher-Quoten das Ergebnis vorher? (`betting_offices.csv` ab 2015)